In [0]:
import os
source_volume_path = "/Volumes/formula1/bronze/data_for_micro_batch"
landing_zone_base = "/Volumes/formula1/bronze/raw_data"

files = dbutils.fs.ls(source_volume_path)

for file_info in files:
    file_name = file_info.name.replace(".csv", "")

    target_path = f"{landing_zone_base}/{file_name}"
    

    temp_df = spark.read.format("csv").option("header", "true").load(file_info.path)
    if temp_df.count() < 200:
        fract = 0.5
    else:
        fract = 0.1
    sampled_df = temp_df.sample(withReplacement=False, fraction=fract)
    
    if sampled_df.count() == 0:
        sampled_df = temp_df.limit(1)

    sampled_df.write \
        .format("csv") \
        .mode("append") \
        .option("header", "true") \
        .save(target_path)
    